# PACE Phase 14B — AMPScanner v2 independently pretrained benchmark

**Analysis-only experiment.** Do not edit the original PACE manuscript, the reviewer comments, or supplementary inputs.

This is a new, independently trained **AMPScanner v2 original-paper pretrained** classifier comparison against PACE Phase 8's saved AMP recognition scores. The 2018 original model is used because its six public AMP/decoy train/validation/test FASTAs can be screened for **exact overlap** before scoring; this does *not* guarantee exclusion of all non-exact homologues or unknown historic source overlap.

The PACE held-out set contains 1,568 peptides; external set contains 1,848 peptides. The notebook excludes any peptide <10 aa, unsupported amino-acid sequence, or exact sequence present in the original AMPScanner 2018 public development datasets. It independently downloads published model weights at the pinned repository revision, records their SHA-256, and uses the original published AA integer mapping and left-padding to 200 positions.

**Important:** the historical model is from TF1/Keras2. If modern Colab cannot load the original `.h5`, the runner exports a *diagnostic ZIP rather than invented scores*.

Results include per-sequence probabilities for both models, paired ROC-AUC differences and bootstrap intervals, exclusion counts, and all model/source hashes. Model weights are omitted from the output ZIP (they remain at the pinned official URL).

Run all cells and upload `PACE_Phase14B_AMPScanner_External_Results.zip` in the existing conversation.

Official source: https://github.com/dan-veltri/amp-scanner-v2 — Veltri et al., *Bioinformatics* (2018), DOI 10.1093/bioinformatics/bty179.


In [ ]:
# Run once; TensorFlow may already be preinstalled in Colab.
# tf-keras permits loading older Keras H5 weights under TensorFlow 2.x when compatible.
%pip -q install tf-keras biopython h5py scikit-learn


In [ ]:
# Phase 14B — AMPScanner v2 vs PACE. No dataset upload is needed.
# All 3,416 PACE heldout/external peptides and scores are embedded from previously verified Phase 8 ZIP.
import os, sys, base64, gzip, hashlib
from pathlib import Path
os.environ['TF_USE_LEGACY_KERAS']='1'
R=Path('/content')
files_to_create=[
    ('phase14b_ampscanner_external_runner.py','H4sIAAE/+GoC/506a1fcOLLf+RXO7AfZidt0NwMDJs45DIGdnIXA5TF7d3v6+KhtuVuD23YsGehw+e+3SpJf3ZBkhpOALZVKpXpXyf94s12JcnvGs22W3VvFSi7ybGfrp59+umI0HuRZurIuj45PrChfFrTkIs8smVtFNUu5WLDYOjq/vI5olrHSuh9becnnPKOplVRpai3zmKXe1hX7UvGSCYtnkpUZk5Z9nKd05lg0iy1qLWnGEyakNat4Kq2kzJd6z8sFFWzfWrA0ziu5zR5xOSCHl0GeDJI8ja2iZDGPJM8z4W19ZvdAhywZlaJDGRHWeDjat0pW5ILLvFwBDOUZz+bb7J6m2xJ3Pz26vjkSFoV/mVVlMq8iPKCaE0x6yJQtvizyUlq5cMVKuJG4d3nu/glccYHWRcpn7ldeJDxlLuwQsRmN7twoT1OmSXQlXzK3SKlM8nK5pY5aUIkLLYP6El71REwlQ/h6pn53Lfz9Nc+YhqvKFNZ7JbAZaTXQMJoXLNMg4i5ltMy8JZMlj0QNU+ZRSKsoFFFeAloK3KNzFgJPIy6A3npiRlOaRSwOaRRVcK5VPbGkUi7YgwjhtYxylgBsyVmp58M0F6JmWVYtixVyNyu2Lm5vAjynTbajHJQik9sFcCssUOCjn2chXRbCyM45BGhveRfz0gYNBFgR3JQVc9kjFzLM79Sbs3V+9PnT6cn1JuIaZ6sPIQq9oiiRsFY+D2RJnK2rk+vbs5vwv58u1/GgRoaXGtmvXWQnRi3DKyaqVAoPNAAw3V6dhVcXFzcBWUhZCH97u6QP3pzLRTWrBCsNXg8Mazum2eCepSCbbTj5wBx9cD8mW9cXt1ew8dXJaUAOdnaGu2M23tnb3dsZJexgZ3iwv38wHo2HjO3u7LLhjO7sjyjZurk6+vT59NPZyXUwIUCrJ0svocRVz3j69k0y/fzx5PjiPw2cfmshzayCnW6dX3w8OQsRf0AujMlf0gKYMdwfHYx+CU9vz85CBeUtdsnW0fmnzxcB+d+j448np//87dO/zs4/X/7P1fXN7//+Dxzx+OLqJDy+OLs9/4z0IqcPhuHJDex7cn0+1k+/rUCzYv18DM4ILRkEOB7poY5AaieENg/EbsUsscSCjnf37Jnjl0xWZWYZc/WaCW/BHmM+B1WwHbUkzh+yNKexXbLU8bcs+AGbChLyVIv2efupFQ+8AOAzUYCyXOkV+PMAMq/N0Ya/ygmADwv2hw4aBLjGAnwD8y2jE0E9AjYN+zsNJkP7EzGAxDcPLgG8xEfkRJ+I+OZkBsJxyWwFvoz4KVBRDz4r1OwxYoW0TtQf4CkSxVr6wVUKZl2BSwS6T8oyL+0ERFCB981y2fDJKjiwP7byJOERBzct8qqMmPUEVD371pNcFcxmjheGGV2yMMQx9kwc7fCZllNChaQh2LlgKKwtfWpwJrEIJtND9QuXB0QzGtyolfIMfGRmzbwYIGNmk0omg/2B4HPieKJIuUQQYTvtmXAgwF+eAKsrOjzmiWXGaSkFys4mH0hnqQFCKvyaNo8WINzYtnHUJcT7M+eZLRyvgvHSdhynt1ydAHeZjPypptB2JkN9vgaSpYYWv8GPbxrV36TAaJBZpHkOni98KLlkNoYit8wfBMYsEXyGEFPLAAaBYjAOfGpJyKWaUpEcX3CdbzTmd3CyRl8IWxYSoi6dpUzZA6i/BdJKwaHHuKhaZoJotMpclPNFchxPmQ15IG7GHpTYCHFZBoKG8B1oURNlR0nH5AI4lPcREoN/48FAYd2EQxqBzBEBEmkpxTHnAdY7zuGDp7iwAJtDjtXvCGBOrdkFzqeSLAQFkcKmrspyDJtWQVZ4tCzpyp5AtmM/TggkOiwlU0ep6iPqKZ06h0UHMAHrQVCFaB1QoaUCAoa0hu+DwluCWB31QB/xYaR4j0YNGKuMQxJgrxwnCMZdgWOOZGcBglHHzTDABkjgyhPVErQDhrI8q0ft0WDl1DNXF8fh0e1xoMnspQz2yi0A4vKqA/BKEmFAf60ziSOTSZhVr2QYuOpD4O3CyvPjeoeNpKMD9StmHzXOtUxEk2CkOMtzCaZPC3WamCcJylIwFgejn4d74+F4z4V8UQS7w2FtA6BvwOMS+J0vwdckFMJ9CKM2LnMOf1j4M0giXpD/6zHsJZ2AAIg4HhYM2RQEI+VAMjbvjQ5x1CiCyk5qB4MIdU5PISFGzL0wPPEHo2lrT19eILddvU5fPhPBup58cQbrQ8gGYEbP5yGeEPEAl+fMRgmsu95HpAUCWATpcAb/7QmIwIsWOY/Ag0FmjkoOfx2UXwpppU4Q3Q4UcElBwd81qGnfUc8av7pG/IQ/Tt0v+HvjXGoKz6ZmW3xp7i440v6lohBKUwhwwp14w/Gu6x38sgt8MyKqt1Q223I5aB8Veu0LX9MZF1UalACMKuRZ2DdiEJDjHn862AW7eGjsJc3NoAoZZnDBHbe1FWURSihr4UQRri0Lc4EwZXM0YuXStB/XUvyH9Rs4XaA0ggThX+AphK4SIYKUTEWQeQVnhOwES69cIUN1AN9v3bBM5OUpkLytVnrW5xy2HtS1HLr0hN4xCwUB756OUrr8kGoxnOkBQ4VM1FzM6TzLhYSSKHgiLQjxZQJ5Cngy5cBCnSYhLTBSq2svx6t3ScI7JK0jdLWmic6khiBu/eRpBniKbzqYtErToXDSrp0GzeIOld/I5V5EE2qiQ4YhGnCCXEvbpBe9w716iFBV7DzDw3jfPsxfoctg/T5hhuvfYbma3iHuqwQe9sioWfy3+ftd5qKTu2MrV1OK6m1obk/WOyb+LAMNY4MhaoNy0R+AGwlOaSpY32/16NEL1YlWr0MpXoCrgNwmhNKhYLAA91p6nTHne+shr9tE0B3sYzD+Y+l2ULX573dZDbUYnOk5VMKsOQ2DL5UaSApzJv7BcDg1VdoLhQ0Bl6L8rOSYqmoPZeSEYmsbX603skbeowV1rvZkvkXeYUPIi6tlIewOuXXqUVZZXYqYxK7uXXhchNg8sh03IedcCPRqT/Xss2fdAr2Q+1jg/ZMBFnHg+YAW1U5D78nAVd959ekedMZep8JXOrFtNlOJdV0YGEowKsJCSB93fh7tuXb9DpX4Y8EiCZuNdveanpz1brT/875Vd+bIBq4nzGtUgUOmLj5ikwpiUi+ngQ2eYcd6ry6Op3Z5Hz4Ah22IgOq/IUB76joShljlB6YQbs6NRXWoimHbHL4o83vIJYAuwAp5/ZwRn+jyNazLW9jFjJQsIX5b+LtEG0jdSQrrArxHhUsibEtChlvJiPh1L8/L8ge7bud5MOWADuTYHITY6zRbQgkNlfvTs0vqYIdU6PBuJijstBKgQBD9CqS/bsx0W7QFNmk6zVnrXtRdVtVxNRx917RaVU4jDkHCNJJN19SqxYg2mlboHoVhfQ0SCCbt1t0lKLe2K9X6NnRlYLI1rwbAGCBHbj8lxkiVn5dsGTT9GOX9mrmaFIjg8KgTSBefYMNuNwFxTJrGydS0PvrSn/T4PZ3gTtPg6c6/1z7bvUesiMnDX6A/WATfvQkavC9jfUlo00kCqEmmGzLNKUDmuogLN2aeO9FB4/u/oJndenXPGiQEvBDGMnCRiLee15w059biVk0W0McUZAJesFsy1Pbnd/kf9Az7ELDPQaiGeCx3oKyjASZ3qBQ49l6ph2oLOl1/r3UpIHdgFlm3yYo5bRize5bmxRIYHcJBy5QWBAVgpN1oJ8R3ZtnARwGZLARhhvko0hSmcjRUS/T7+9GwBubZPU05EAr1dQ5VKI8VnKZdAxHSSczK/CF4evv20d2kVdEB2PDmoKHTx4pwjVAHHZc5M6gGhahB/GbkuduSakb9jmDa2uShw0Ug1e+JcwOsbfhc3N5s1zpvKA1bktJ8rtribmdP3RnSnRP6gI2Td5MfZMELp506G3ob5RCPIQdkCdbtzQrQWlMPNZcp3jGCQkQr+zGiNDrqvIqcJhJvAf4u7h53X9hFIY4ZBpnmoN/cYZMv7WYd3jtN303ZVo8KUOWh/1JCc5PnVgLVk5HCICkZa53moYUSwy5eBWkYo6iWEB46130mptfpYZK3jlj7GBYPdE69Td61FwObXOm71hZy08G2e33bzSr17Wyp+3UmtLdIul5/kyoV7acBKSAX1cfRZYFKKkl7dpOth0smabBZ466RsrFPB+c06KAyVfHvrOTJqlsRE2HFeVShs2NYAkOKYbyWNR4OXVBwSARLyAZpZkXwa46ej0thqTTbEpBGZjJdmSI4E4G+je0k87UycUgzgQ1Apg0PLtq24+MK+F93jzopHYxid1G1HRFiMJqi0GykSnWMIXe9zbDDPM/4V6BdZxr6BE+w4tkFd2aSSRtXuHgiR10Pz7FyM1ynqg3xRKnPlXZwFzwx7MQy4EqJnR8dQMwlBraEWYwNlq+szIVtbxqJi/u4MZYHCAcb7ozbHIW7ZR99f3Eb8vD2kIlOS0yTOom02UbK/3RD4vTFPTtUT7g7QHI1ZsefBvqp28FoLlDDrDDqBccNkQqh6dASNjfitkHtzqiMFqEAUQSj8b4LbmCWCxYMHQcyYaUI9mDU8yyIwXkTbPIPOyzoLHBTLhIIYnhpgNAeTVNbzeOraVIPRmyw245hv/rD6B0OfvNyiZwrhdFVI17UWVXWKIzWbkVHNgADm9EZB/e8MkLB6gsrPUWVp4Bd3chq6XJ6I0iVU1dByutqJ/SVF/3Tu+qgfvmNPq3pmd2/GGY7q9Rd8hLlAt6m+wWDirZ9nuv0rMnLLNWq7qZkKjwhxZOXCqJOE1cBVzPIvFRZDJE8i9UdqlpsExBhWIca4qZ0OYupVfq6e2oTk0KNhjLfGXbmR8P36tIAGKNBwM2+D3aGnTU7I5nvddfsjF5aszd0pv2uBw0m5StxF5W1Dc5BoNmATqk5l1060/VrO30D8n5H6aVOUaVdk9Lt1qsdqQN2MFa3rDyrWA8ZAi03Gud+P+dSlqhI0wS6WgBBVw7KbIOl+/btxtXS+sUh7qn6oT98fWHvDocqR243DIK+qHV2i65xrdWuezyA+q+cprlkevs2dn7IEO4x9nGMu+rrlLDNPYw9dAzhO6gKykvMu2ruaATqDN8I/bhhyiQjh93prGuZpmjqm+bhZoB/YHy+gAyTZ2FVqDAfh/g9yFR36dq0hbRrQ1BaTPxivKQW3p8q6TK5jARDtjvtpHaVy7MYvxUY91vxLYD58GGZ37G63YTd8WDyZBTdJ6NvfDjhE28fptuPMWDgoPNFhplvP8uAgV9e+jYDxvfIs9vuOvz2rqP1Xcf9Xcfru45f2XVMnnvJS9+6kBev0+FMiLkyQd8y8obrWVDvQwXygf6RHR0d/5F9mP2RHR/DA2mvYDur0Nm88hmMgh8ZJUWHRK5XGaR4kkeWKk3V90rqczRhFYgz9ojpKBY0urPxWJVwVRu0e3Vf0BUqYtvZMoBEN0x99cclGH9hZln8tQaV/mKQ+PU3bZ4eqLvmttNJ1QkUGaHe/TtarkleV3H1ZYD5vs77Ly9OsVXafrHl4kcCzfSnyxC4fHZ0c/JRdcrBYEWKDYTgF/W1wNd+YEx0gMFmAdLqlfM0n9nkLXE2P/1I2j6tCjiJh57vTdDm/771VR8NPzyANAuYw+/hnDnibipGJWX91ZgF9PrE7RymfcRPUYDXHrYWIYurP+QB2QMtdYsbnHoIKQX4nZD4dTJHBgNl/0QdbgV+vJzfd1o32jcc4gx75NIevnDHIgLVrz5UOta6SnR+WkurCOpI8cb6yAV+cyKbbrAVpaCm6NpLawbZ6GJJyzsLs0ZsoHvE+ZGPj+QsaL6k9LTaYeEOBBmdSihPq5KFLZR8lH3NkjOn02/EY+AiFkNoUZcC9VnkbDLY2YE6etrhCaTG/w+g5mFiJSsAAA==','94eaeaa37b962d9449d08e9fc7d35daca86f15cc85762c4f7ab1498b053977a2'),
    ('phase14b_AMPScanner_evaluation_manifest.csv','...'),
]
for name,encoded,sha in files_to_create:
    data=gzip.decompress(base64.b64decode(encoded))
    assert hashlib.sha256(data).hexdigest()==sha,(name,'SHA-256 check failed')
    (R/name).write_bytes(data)
    print(name,len(data),'bytes, SHA-256 verified')
print('No original author files are changed.')


In [ ]:
# Run original published AMPScanner 2018 checkpoint on exactly the filtered PACE sequences.
# This downloads official upstream FASTAs and model, excludes exact training/development overlaps, then scores.
import subprocess,sys,os
os.environ['TF_USE_LEGACY_KERAS']='1'
p=subprocess.run([sys.executable,'/content/phase14b_ampscanner_external_runner.py'],check=False)
print('Run exit code:',p.returncode)
from google.colab import files
from pathlib import Path
result=Path('/content/PACE_Phase14B_AMPScanner_External_Results.zip')
if result.exists():
    print('Download the ZIP and upload it back in the same PACE chat.')
    files.download(str(result))
else:
    print('No ZIP generated; share the full traceback from this cell.')
